Cross Validation:
until now our accuracy was dependant on one random split. If split lucky -> high accuracy, otherwise low

cross validation fixes this, it splits data into 5 equal parts (folds). it trains 5 times, each time using 4 folds for training and 1 for testing. Then average all the scores, giving us reliable, unbiased estimates of how the model actually performs.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('Titanic-Dataset.csv')
df.drop(columns = ['Cabin', 'Name', 'Ticket', 'PassengerId'], inplace = True)
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df['Sex'] = df['Sex'].map({'male' : 0, 'female' : 1})
df = pd.get_dummies(df, columns = ['Embarked'], drop_first = True)

X = df.drop(columns = ['Survived'])
y = df['Survived']

# Scale for KNN
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('ready')

KFold : splitting data into multiple folds

shuffle=true : randomly shifting data before splitting so folds aren't in biased order

cross_val_score returns an array of 5 scores - one per fold. we take mean of overall accuracy and std to see how consistent the model is.

In [ ]:
# defining 5 fold cross validation
kf = KFold(n_splits = 5, shuffle = True, random_state = 42)

#define all 4 models
models = {
    'Logistic Regression' : LogisticRegression(max_iter = 1000),
    'Decision Tree' : DecisionTreeClassifier(max_depth = 6, random_state = 42),
    'Random Forest' : RandomForestClassifier(n_estimators = 100, random_state = 42),
    'KNN (KNN = 17)' : KNeighborsClassifier(n_neighbors = 17),
}
results = {}

for name, model in models.items():
    # cross val score trains and tests automatically across all 5 folds
    scores = cross_val_score(model, X_scaled, y, cv = kf, scoring = 'accuracy')
    results[name] = scores
    print(f"{name}")
    print(f" Scores: {scores.round(4)}")
    print(f" Mean : {scores.mean():.4f} | Std : {scores.std():.4f}")


In [ ]:
# Visualize
# collect means and stds for plotting

names = list(results.keys())
means = [results[m].mean() for m in names]
stds = [results[m].std() for m in names]

plt.figure(figsize=(10, 5))
bars = plt.bar(names, means, yerr=stds, capsize=6,
               color=['steelblue', 'coral', 'seagreen', 'mediumpurple'])
plt.title('5-Fold Cross Validation — Model Comparison')
plt.ylabel('Accuracy')
plt.ylim(0.6, 1.0)

# add mean score on top of each bar
for i, (mean, std) in enumerate(zip(means, stds)):
    plt.text(i, mean + std + 0.005, f'{mean:.2%}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('cross_val_comparison.png', dpi=150)
plt.show()